# Error Potential (ErrP)

## Overview
This notebook analyzes the error-related negativity (ERN) from ErpCore2021-ERN. It compares averaged ERP for Target (error) vs NonTarget (correct) trials at FCz, Cz, Fz.

## What to look for
- Negative deflection appears 80-150 ms after erroneous response
- Negative response is clear at FCz

## 1. Install dependencies

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn

## 2. Load data

In [ ]:
import numpy as np
from moabb.datasets import ErpCore2021_ERN
from moabb.paradigms import P300

FS = 1024
dataset = ErpCore2021_ERN()
paradigm = P300(fmin=0.5, fmax=40)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

raw = dataset.get_data(subjects=[1])
s1 = raw[1]; sess = list(s1.values())[0]; run = list(sess.values())[0]
ch_names = run.ch_names
target_chs = ['FCz', 'Cz', 'Fz']
ch_idx = [ch_names.index(c) for c in target_chs]

target_avg = X[labels == 'Target'].mean(axis=0)
nontarget_avg = X[labels == 'NonTarget'].mean(axis=0)
t = np.arange(X.shape[2]) / FS * 1000
print(f"Data: {X.shape}, Targets: {np.sum(labels=='Target')}, NonTargets: {np.sum(labels=='NonTarget')}")

## 3. Interactive plot

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=target_chs, shared_yaxes=True)
for i, (ch_name, ci) in enumerate(zip(target_chs, ch_idx)):
    fig.add_trace(go.Scatter(x=t, y=target_avg[ci], name='Target (error)', line=dict(color='coral', width=2), showlegend=(i==0)), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=t, y=nontarget_avg[ci], name='NonTarget (correct)', line=dict(color='steelblue', width=1.5), showlegend=(i==0)), row=1, col=i+1)
    fig.add_vrect(x0=80, x1=150, fillcolor='red', opacity=0.15, row=1, col=i+1)
fig.update_layout(title='ErrP - ErpCore2021-ERN Subject 1', width=1200, height=400)
fig.update_xaxes(title_text='Time (ms)')
fig.update_yaxes(title_text='Amplitude (V)', row=1, col=1)
fig.show()

## Summary
- ErrP is a negative response 80-150 ms after error
- Appears at FCz and Cz
- Used to auto-correct BCI errors